# Proprietary Trading Firms Have Convex Payoffs
You don't have to beat the market to profit from proprietary trading firms. You can engineer a solution with skew, because challenges aren't too dissimilar from other structured products.

**Author:** puzzle

**Created:** 2026-03-09

**Modified:** 2026-03-11

### Sources
- [1] Breiman, L. (1961) Optimal Gambling Systems for Favorable Games. Proceedings of the 4th Berkeley Symposium on Mathematical Statistics and Probability, 1, 65-78. Available at: https://projecteuclid.org/ebooks/berkeley-symposium-on-mathematical-statistics-and-probability/Proceedings-of-the-Fourth-Berkeley-Symposium-on-Mathematical-Statistics-and/chapter/Optimal-Gambling-Systems-for-Favorable-Games/bsmsp/1200512159.pdf (Accessed 11th March 2026).


In [1]:
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.express as px
import plotly.graph_objects as go

In [2]:
from jfmi.plot.utilities import load_plotly_templates

In [3]:
from jfmi.plot.templates import COLOURS

In [4]:
load_plotly_templates()

## Explore Toy Strategies


### Plot Strategy Parameter Space
The expected value of a random variable is: $E[X] = \sum_{i} p_{i} x_{i}$, where $x_{1}, x_{2}, \ldots$ are the possible outcomes of the random variable $X$, and $p_{1}, p_{2}, \ldots$ are their corresponding probabilities.

In this case, we win $R$ and lose $1$ with probability $W$.

So: $E[X] = W \cdot R + (1 - W) \cdot (-1) = W \cdot R - (1 - W)$

At: $E[X] = 0$, $R = (1 - W) / W$


In [5]:
arr_win_rate = np.arange(0.0, 1.0 + 0.05, 0.05)
arr_risk_to_reward = np.arange(0.00, 5.00 + 0.25, 0.25)
# arr_win_rate = np.arange(0.5, 1.0 + 0.025, 0.025)
# arr_risk_to_reward = np.arange(0.00, 1.00 + 0.05, 0.05)

In [6]:
# Calculate the expected value element-wise across every combination.
W, R = np.meshgrid(arr_win_rate, arr_risk_to_reward)

EV = W * R - (1 - W)  # normalised losses

# Contour plot for EV
fig = go.Figure()

fig.add_trace(
    go.Contour(
        z=EV,
        x=arr_win_rate,
        y=arr_risk_to_reward,
        colorscale=[
            [0, COLOURS["red"][5]],
            [1 / 6, COLOURS["white"]],
            [1, COLOURS["green"][5]],
        ],
        contours=dict(
            start=-1,
            end=5,
            size=0.2,
            showlabels=True,
        ),
        colorbar=dict(title="EV"),
        hovertemplate=(
            "<b>Win Rate:</b> %{x:.2f}<br>"
            "<b>Risk : Reward:</b> %{y:.2f}<br>"
            "<b>EV:</b> %{z:.2f}<br>"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Expected Value Heatmap",
    xaxis_title="Win Rate",
    yaxis_title="Risk : Reward",
    margin=dict(t=60),
    width=800,
    height=600,
)

fig.show()

### Estimate Strategy Pass Rates


In [7]:
threshold = 1
mask = (-threshold <= EV) & (threshold >= EV)
# mask = EV == 0

df_ev_zero = pd.DataFrame(
    {
        "Win Rate": W[mask],
        "Risk : Reward": R[mask],
        "EV": EV[mask],
    }
)
# df_ev_zero = pd.DataFrame(
#     {
#         "Win Rate": [1 / 5, 1 / 4, 1 / 3, 1 / 2, 2 / 3, 1],
#         "Risk : Reward": [4, 3, 2, 1, 1 / 2, 0],
#         "EV": [0, 0, 0, 0, 0, 0],
#     }
# )

df_ev_zero = df_ev_zero[~(df_ev_zero["Risk : Reward"] == 0)]

df_ev_zero

,Win Rate,Risk : Reward,EV
21,0.00,0.25,-1.0000
22,0.05,0.25,-0.9375
23,0.10,0.25,-0.8750
24,0.15,0.25,-0.8125
25,0.20,0.25,-0.7500
...,...,...,...
277,0.10,5.00,-0.4000
278,0.15,5.00,-0.1000
279,0.20,5.00,0.2000
280,0.25,5.00,0.5000


In [8]:
sr_risk = pd.Series(np.arange(0.5, 2.0 + 0.25, 0.25) / 100, name="Risk")

df_monte_carlo = df_ev_zero.merge(sr_risk.to_frame(), how="cross")

df_monte_carlo

,Win Rate,Risk : Reward,EV,Risk
0,0.0,0.25,-1.0,0.0050
1,0.0,0.25,-1.0,0.0075
2,0.0,0.25,-1.0,0.0100
3,0.0,0.25,-1.0,0.0125
4,0.0,0.25,-1.0,0.0150
...,...,...,...,...
1822,0.3,5.00,0.8,0.0100
1823,0.3,5.00,0.8,0.0125
1824,0.3,5.00,0.8,0.0150
1825,0.3,5.00,0.8,0.0175


In [9]:
def constant_risk_management(
    account_equity,
    initial_account_equity,
    pass_level,
    fail_level,
    base_risk_to_reward,
    base_risk,
):
    """Do not adjust the risk."""
    return base_risk

In [10]:
def boundary_risk_management(
    account_equity,
    initial_account_equity,
    pass_level,
    fail_level,
    base_risk_to_reward,
    base_risk,
    max_risk_upper=0.025,
    max_risk_lower=0.030,
    min_risk=0.005,
    buffer=0.001,
):
    """Adjust the risk based on the proximity to pass/fail levels.

    This is only applicable if the account doesn't fail when its unrealised losses touch
    the fail level (i.e. if passing/failing is determined by realised gains/losses).

    These conditions mean that this risk management strategy isn't dissimilar to poker
    tournament play. During the bubble, players with big stacks (who're likely to make
    it through) tend to tighten up, whilst players with short stacks tend to become more
    aggressive. Similarly, if you're ever drawing down badly (to perhaps <20 big blinds)
    you tend to become more aggressive. If it's really bad (maybe <10 big blinds),
    you're going to get blinded out, so you're shoving with basically anything.
    """
    pass_equity = initial_account_equity * (1 + pass_level)
    fail_equity = initial_account_equity * (1 - fail_level)

    equity_to_pass = pass_equity - account_equity
    equity_to_fail = -(fail_equity - account_equity)

    # Solve next_equity = account_equity * (1 + risk * base_risk_to_reward) for risk.
    risk_to_pass = equity_to_pass / (account_equity * base_risk_to_reward) + buffer
    risk_to_fail = equity_to_fail / account_equity - buffer

    # If you're going to miss the pass level by a hair, increase your risk to get there.
    if base_risk < risk_to_pass <= max_risk_upper:
        adjusted_risk = risk_to_pass
    # If you're going to cross the fail level by a hair, decrease your risk to avoid it.
    elif min_risk <= risk_to_fail < base_risk:
        adjusted_risk = risk_to_fail
    # If you're going to fail regardless, you might as well risk it all.
    elif risk_to_fail < min_risk:
        adjusted_risk = max_risk_lower
    else:
        adjusted_risk = base_risk

    return adjusted_risk

In [11]:
n_simulations = 1000

pass_level = 10.0 / 100  # fraction profit (from initial balance)
fail_level = 6.0 / 100  # fraction drawdown (from initial balance)

initial_account_equity = 1.0

dict_results = {}

for idx, row in df_monte_carlo.iterrows():
    dict_results[idx] = []

    for _ in range(n_simulations):
        account_equity = initial_account_equity

        path = [initial_account_equity]

        while True:
            # adjusted_risk = boundary_risk_management(
            adjusted_risk = constant_risk_management(
                account_equity,
                initial_account_equity,
                pass_level,
                fail_level,
                row["Risk : Reward"],
                row["Risk"],
                # max_risk_upper=row["Risk"] + 0.005,
            )

            # Uniformly randomly sample a number and compare it with the win rate to
            # determine if the trade is a winner or loser.
            if np.random.random() < row["Win Rate"]:
                account_equity *= 1 + adjusted_risk * row["Risk : Reward"]
            else:
                account_equity *= 1 - adjusted_risk

            path.append(account_equity)

            if account_equity >= initial_account_equity * (1 + pass_level):
                break
            if account_equity <= initial_account_equity * (1 - fail_level):
                break

        dict_results[idx].append(path)

In [12]:
df_monte_carlo["Pass Rate"] = [
    sum(path[-1] >= 1 + pass_level for path in dict_results[idx])
    / len(dict_results[idx])
    for idx in df_monte_carlo.index
]

df_monte_carlo["Pass Rate SE"] = [
    np.sqrt(p * (1 - p) / len(dict_results[idx]))
    for p, idx in zip(df_monte_carlo["Pass Rate"], df_monte_carlo.index, strict=True)
]

# For a 95% confidence interval, we can use the critical value from the standard
# normal distribution, $Z_{0.975} \approx 1.96$ (where 2.5% is in each tail).
df_monte_carlo["Pass Rate CI Low"] = (
    df_monte_carlo["Pass Rate"] - 1.96 * df_monte_carlo["Pass Rate SE"]
).clip(0, 1)
df_monte_carlo["Pass Rate CI High"] = (
    df_monte_carlo["Pass Rate"] + 1.96 * df_monte_carlo["Pass Rate SE"]
).clip(0, 1)

df_monte_carlo

,Win Rate,Risk : Reward,EV,Risk,Pass Rate,Pass Rate SE,Pass Rate CI Low,Pass Rate CI High
0,0.0,0.25,-1.0,0.0050,0.000,0.000000,0.000000,0.000000
1,0.0,0.25,-1.0,0.0075,0.000,0.000000,0.000000,0.000000
2,0.0,0.25,-1.0,0.0100,0.000,0.000000,0.000000,0.000000
3,0.0,0.25,-1.0,0.0125,0.000,0.000000,0.000000,0.000000
4,0.0,0.25,-1.0,0.0150,0.000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...
1822,0.3,5.00,0.8,0.0100,0.872,0.010565,0.851293,0.892707
1823,0.3,5.00,0.8,0.0125,0.779,0.013121,0.753283,0.804717
1824,0.3,5.00,0.8,0.0150,0.758,0.013544,0.731454,0.784546
1825,0.3,5.00,0.8,0.0175,0.706,0.014407,0.677762,0.734238


In [13]:
# percentiles = [25, 50, 75]

# for percentile in percentiles:
#     df_monte_carlo[f"Trades to Outcome {percentile}th Percentile"] = [
#         np.percentile([len(path) - 1 for path in dict_results[idx]], percentile)
#         for idx in df_monte_carlo.index
#     ]

#     df_monte_carlo[f"Trades to Outcome {percentile}th Percentile SE"] = [
#         np.std([len(path) - 1 for path in dict_results[idx]], ddof=1)
#         / np.sqrt(len(dict_results[idx]))
#         for idx in df_monte_carlo.index
#     ]

#     df_monte_carlo[f"Trades to Outcome {percentile}th Percentile CI Low"] = (
#         df_monte_carlo[f"Trades to Outcome {percentile}th Percentile"]
#         - 1.96 * df_monte_carlo[f"Trades to Outcome {percentile}th Percentile SE"]
#     )
#     df_monte_carlo[f"Trades to Outcome {percentile}th Percentile CI High"] = (
#         df_monte_carlo[f"Trades to Outcome {percentile}th Percentile"]
#         + 1.96 * df_monte_carlo[f"Trades to Outcome {percentile}th Percentile SE"]
#     )

# df_monte_carlo

In [14]:
def bootstrap_percentile_ci(data, percentile, n_boot=1000):
    """Bootstrap percentile confidence intervals.

    Randomly resample the data with replacement to compute the statistic lots of times,
    then take the 95% confidence intervals of the percentiles. This can be useful when
    distributions are non-normal and analytical solutions are difficult.
    """
    point = np.percentile(data, percentile)

    boot = []

    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot.append(np.percentile(sample, percentile))

    # Take the 95% confidence interval.
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    return point, ci_low, ci_high

In [15]:
percentiles = [25, 50, 75]

outcomes = {
    "Pass": lambda path: path[-1] >= 1 + pass_level,
    "Fail": lambda path: path[-1] <= 1 - fail_level,
}

for outcome, condition in outcomes.items():
    for percentile in percentiles:
        points = []
        ci_lows = []
        ci_highs = []

        for idx in df_monte_carlo.index:
            trades = [len(path) - 1 for path in dict_results[idx] if condition(path)]

            if len(trades) == 0:
                point, ci_low, ci_high = np.nan, np.nan, np.nan
            else:
                point, ci_low, ci_high = bootstrap_percentile_ci(
                    trades, percentile, n_boot=250
                )

            points.append(point)
            ci_lows.append(ci_low)
            ci_highs.append(ci_high)

        df_monte_carlo[f"Trades to {outcome} {percentile}th Percentile"] = points
        df_monte_carlo[f"Trades to {outcome} {percentile}th Percentile CI Low"] = (
            ci_lows
        )
        df_monte_carlo[f"Trades to {outcome} {percentile}th Percentile CI High"] = (
            ci_highs
        )

df_monte_carlo

,Win Rate,Risk : Reward,EV,Risk,Pass Rate,Pass Rate SE,Pass Rate CI Low,Pass Rate CI High,Trades to Pass 25th Percentile,Trades to Pass 25th Percentile CI Low,...,Trades to Pass 75th Percentile CI High,Trades to Fail 25th Percentile,Trades to Fail 25th Percentile CI Low,Trades to Fail 25th Percentile CI High,Trades to Fail 50th Percentile,Trades to Fail 50th Percentile CI Low,Trades to Fail 50th Percentile CI High,Trades to Fail 75th Percentile,Trades to Fail 75th Percentile CI Low,Trades to Fail 75th Percentile CI High
0,0.0,0.25,-1.0,0.0050,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,13.0,13.0,13.0,13.0,13.0,13.0,13.0,13.0,13.0
1,0.0,0.25,-1.0,0.0075,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
2,0.0,0.25,-1.0,0.0100,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0,7.0
3,0.0,0.25,-1.0,0.0125,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
4,0.0,0.25,-1.0,0.0150,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1822,0.3,5.00,0.8,0.0100,0.872,0.010565,0.851293,0.892707,6.0,6.0,...,16.0,7.0,7.0,7.0,7.0,7.0,7.0,13.0,13.0,18.0
1823,0.3,5.00,0.8,0.0125,0.779,0.013121,0.753283,0.804717,3.0,3.0,...,12.0,5.0,5.0,5.0,5.0,5.0,5.0,11.0,11.0,11.0
1824,0.3,5.00,0.8,0.0150,0.758,0.013544,0.731454,0.784546,3.0,3.0,...,9.0,5.0,5.0,5.0,5.0,5.0,5.0,10.0,10.0,10.0
1825,0.3,5.00,0.8,0.0175,0.706,0.014407,0.677762,0.734238,3.0,3.0,...,6.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0


The trades to outcome explodes at lower risk : reward and lower risk values.


In [16]:
def get_max_drawdown(path):
    """A smaller utility than those in my shared library."""
    peak = np.maximum.accumulate(path)
    drawdown = (path - peak) / peak

    return np.abs(drawdown.min())

In [17]:
percentiles = [25, 50, 75]

for percentile in percentiles:
    points = []
    ci_lows = []
    ci_highs = []

    for idx in df_monte_carlo.index:
        drawdowns = [get_max_drawdown(path) for path in dict_results[idx]]

        if len(trades) == 0:
            point, ci_low, ci_high = np.nan, np.nan, np.nan
        else:
            point, ci_low, ci_high = bootstrap_percentile_ci(
                trades, percentile, n_boot=250
            )

        points.append(point)
        ci_lows.append(ci_low)
        ci_highs.append(ci_high)

    df_monte_carlo[f"Max Drawdown {percentile}th Percentile"] = points
    df_monte_carlo[f"Max Drawdown {percentile}th Percentile CI Low"] = ci_lows
    df_monte_carlo[f"Max Drawdown {percentile}th Percentile CI High"] = ci_highs

df_monte_carlo

,Win Rate,Risk : Reward,EV,Risk,Pass Rate,Pass Rate SE,Pass Rate CI Low,Pass Rate CI High,Trades to Pass 25th Percentile,Trades to Pass 25th Percentile CI Low,...,Trades to Fail 75th Percentile CI High,Max Drawdown 25th Percentile,Max Drawdown 25th Percentile CI Low,Max Drawdown 25th Percentile CI High,Max Drawdown 50th Percentile,Max Drawdown 50th Percentile CI Low,Max Drawdown 50th Percentile CI High,Max Drawdown 75th Percentile,Max Drawdown 75th Percentile CI Low,Max Drawdown 75th Percentile CI High
0,0.0,0.25,-1.0,0.0050,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,13.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
1,0.0,0.25,-1.0,0.0075,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,9.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
2,0.0,0.25,-1.0,0.0100,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,7.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
3,0.0,0.25,-1.0,0.0125,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,5.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
4,0.0,0.25,-1.0,0.0150,0.000,0.000000,0.000000,0.000000,NaN,NaN,...,5.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1822,0.3,5.00,0.8,0.0100,0.872,0.010565,0.851293,0.892707,6.0,6.0,...,18.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
1823,0.3,5.00,0.8,0.0125,0.779,0.013121,0.753283,0.804717,3.0,3.0,...,11.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
1824,0.3,5.00,0.8,0.0150,0.758,0.013544,0.731454,0.784546,3.0,3.0,...,10.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
1825,0.3,5.00,0.8,0.0175,0.706,0.014407,0.677762,0.734238,3.0,3.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0


Maximum drawdowns can be greater than the maximum loss because they're measured from the peak.

Visually, the 50th percentile is approximately constant across all strategies. For strategies that take fewer trades, the 25th percentile is actually smaller (closer to the maximum loss), presumably because they immediately smash into the boundaries. I don't immediately see a pattern in the 75th percentile.


### Plot Strategy Pass Rate Surface


In [18]:
continuous_colour_scale = px.colors.sequential.Viridis

In [19]:
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=df_monte_carlo["Risk : Reward"],
            y=df_monte_carlo["Win Rate"],
            z=df_monte_carlo["Pass Rate"],
            mode="markers",
            marker=dict(
                size=4,
                color=df_monte_carlo["Risk"],
                colorscale=continuous_colour_scale,
                colorbar=dict(title="Risk"),
            ),
            hovertemplate=(
                "<b>Risk : Reward :</b> %{x:.3f}<br>"
                "<b>Win Rate:</b> %{y:.3f}<br>"
                "<b>Pass Rate:</b> %{z:.3f}<br>"
                "<b>Risk:</b> %{marker.color:.3f}<br>"
                "<extra></extra>"
            ),
        )
    ]
)

fig.update_layout(
    title="Toy Strategy Pass Rate Scatter Plot",
    scene=dict(
        xaxis_title="Risk : Reward",
        yaxis_title="Win Rate",
        zaxis_title="Pass Rate",
    ),
    width=800,
    height=600,
)

fig.show()

Generally, increasing the risk per trade increases the variance, which decreases the pass rate. This effect is generally at win rates of 0.3-0.5, and grows towards with the risk:reward ratio. Presumably, even if your strategy has a high expected value, if you risk too much you're going to be likely to tag the boundaries.


### Plot Monte Carlo Paths


In [20]:
colours = pc.sample_colorscale(
    continuous_colour_scale, list(np.linspace(0, 1, n_simulations))
)

In [21]:
strategy_idx = df_monte_carlo[
    (df_monte_carlo["Win Rate"] == 0.5)
    & (df_monte_carlo["Risk : Reward"] == 1)
    & (df_monte_carlo["Risk"] == 0.02)
].index[0]

fig = go.Figure()

for idx, path in enumerate(dict_results[strategy_idx]):  # all toy strategy paths
    fig.add_trace(
        go.Scatter(
            y=path,
            mode="lines",
            line=dict(color=colours[idx], width=1),
            opacity=0.2,
        )
    )

fig.add_hline(
    y=initial_account_equity * (1 + pass_level),
    line_color=COLOURS["green"][5],
    line_width=2,
    opacity=0.75,
)

fig.add_hline(
    y=initial_account_equity * (1 - fail_level),
    line_color=COLOURS["red"][5],
    line_width=2,
    opacity=0.75,
)

fig.update_layout(
    title=f"Toy Strategy {strategy_idx} Monte Carlo Paths",
    xaxis_title="Trade Number",
    yaxis_title="Account Equity",
    showlegend=False,
)

fig.show()

This plot isn't beautiful, but it definitely raises the point that you should adjust your risk slightly up or down so that you end up on the right side of the boundary.

With this level of risk, three wins in a row (at ~3R in units of risk) will push you over the profit target. But because a win followed by a loss will take you below your starting equity, that ~3R reduces slightly over time. By trade 9 you need ~4R to pass (or you can increase your risk slightly as the number of trades increases).

Similarly, if you're drawing down to the maximum loss, why would you take a trade that'd leave you awkwardly ~1/2 R above the limit? You might as well increase your risk so you land *just* above the boundary. If you win, you're closer to the profit target. If you lose, you still have another trade to go (and you might as well size that trade up as much as possible). This effect gets bigger the bigger your risk is, so perhaps you counterintuitively want to make your risk as large as feasible within the other rules (daily consistency, daily drawdowns, etc.).

This only applies if passing/failing is determined by realised (not unrealised) gains/losses. I'm guessing that most firms require you to realise your profits to pass, but consider unrealised losses when checking for failure. In this case, your final trade before failing would have to go perfectly in your favour, so you'd probably have to use a different strategy for that trade (for example, retests of the top/bottom of ranges before breakouts, or liquidity grabs at the top/bottom of ranges before reversals).
